# The Normal Approach

Here is the simplest, "Classic" implementation of a Singleton in Python.

In [15]:
class Singleton:
    # 1. The static variable to hold the single instance
    _instance = None

    def __new__(cls, *args, **kwargs):
        # 2. Check if the instance exists
        if cls._instance is None:
            # If not, create it using the parent (object) class
            cls._instance = super().__new__(cls)
        
        # 3. Return the existing instance
        return cls._instance

    def __init__(self, value=None):
        # 4. Check if we have already initialized this object
        # (Because __init__ is called every time you type Singleton(), 
        # even if __new__ returns the same old object)
        if hasattr(self, "_initialized") and self._initialized:
            return
            
        # 5. Actual Initialization logic (runs only once)
        self.value = value
        self._initialized = True

# ==========================================
# CLIENT CODE
# ==========================================

# First creation
s1 = Singleton("Database Connection")
print(f"s1 Value: {s1.value}")

# Second creation (attempting to change value)
s2 = Singleton("New Connection")
print(f"s2 Value: {s2.value}") 

# Verification
print(f"Are they the same object? {s1 is s2}")

s1 Value: Database Connection
s2 Value: Database Connection
Are they the same object? True


#### Explanation of the Logic
- `_instance`: This class-level variable acts as the storage. Initially None.
- `__new__`: This is the allocator. It runs before `__init__`. We intercept it to say: "If we already have an instance, just return that one. Don't make a new one."
- `_initialized`: This is crucial in Python. Even if `__new__` returns the old object, Python still automatically runs `__init__` on it. Without this flag check, `s2 = Singleton("New Connection")` would overwrite `self.value` of `s1`.

#### A Note on Thread Safety
If two threads call `Singleton()` at the exact same millisecond for the first time, they might both pass the `if cls._instance is None` check and create two different objects. That is why the threading.Lock is usually added in production code.

# The Normal Approach (With Thread Safety)

This is the standard way to implement a Singleton class. We override the `__new__` method (which allocates memory) to ensure we only create the object once.

Lazy Initialization: The instance is not created until you call `Database()` for the first time.

In [4]:
import threading

class Singleton:
    _instance = None
    _lock = threading.Lock() # Ensure thread safety

    def __new__(cls, *args, **kwargs):
        # Double-checked locking pattern for thread safety
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        # Note: __init__ runs every time you call Singleton().
        # You often need to add a flag to prevent re-initialization.
        if not hasattr(self, "initialized"):
            self.value = value
            self.initialized = True

# Usage
s1 = Singleton("First")
s2 = Singleton("Second")

print(s1 is s2)       # True
print(s1.value)       # "First" (s2 didn't overwrite because of the flag check)
print(s2.value)       # "First"

True
First
First


Pros:
- Easy to understand for OOP developers.
- Supports inheritance.

Cons:
- __init__ is called every time Singleton() is requested, requiring extra logic to prevent re-setting attributes.
- Thread safety requires manual locking.

# Flow of Execution for Python Singleton

This document details exactly what happens when you create instances of a class using the `__new__` Singleton pattern.

### The Code Reference
We are analyzing this specific implementation:

1.  **`__new__`**: Checks `_instance`. Creates it if missing. Returns it.
2.  **`__init__`**: Checks `_initialized`. Sets value if missing. Returns immediately if present.

---

### Phase 1: The First Creation
**Code Executed:** `s1 = Singleton("Database")`

1.  **Python calls `__new__`**
    * **Check:** Is `cls._instance` None? -> **YES** (It is currently `None`).
    * **Action:** Python allocates memory for a brand new object (Let's call it `Object_A` at address `0x001`).
    * **Storage:** `cls._instance` is set to `Object_A`.
    * **Return:** `Object_A` is returned to the interpreter.

2.  **Python calls `__init__`** (Automatically triggers after `__new__`)
    * **Input:** Python passes `Object_A` into `__init__`.
    * **Check:** Does `Object_A` have `_initialized` attribute? -> **NO**.
    * **Action:** Sets `self.value = "Database"`.
    * **Flag:** Sets `self._initialized = True`.
    * **Result:** `s1` now holds `Object_A` ("Database").

---

### Phase 2: The Second Creation
**Code Executed:** `s2 = Singleton("New Connection")`

1.  **Python calls `__new__`**
    * **Check:** Is `cls._instance` None? -> **NO** (It holds `Object_A`).
    * **Action:** Logic skips the creation step.
    * **Return:** The **EXISTING** `Object_A` is returned.

2.  **Python calls `__init__`** (Automatically triggers again!)
    * *Note: Python doesn't know it's a Singleton. It just sees an object returned, so it tries to initialize it.*
    * **Input:** Python passes `Object_A` into `__init__`.
    * **Check:** Does `Object_A` have `_initialized` attribute? -> **YES** (We set this in Phase 1).
    * **Action:** The `if` statement evaluates to True.
    * **Result:** The method hits `return` immediately.
    * **Crucial Outcome:** The line `self.value = "New Connection"` is **SKIPPED**. `s2` holds `Object_A`, and the value remains "Database".

---

### Visual Trace Table

| Step | User Code | `_instance` Variable | Object in Memory (`0x001`) Value |
| :--- | :--- | :--- | :--- |
| **Start** | (Program Start) | `None` | (Not Created) |
| **1** | `__new__` called for s1 | `0x001` | (Empty Object) |
| **2** | `__init__` called for s1 | `0x001` | `value="Database"` |
| **3** | `__new__` called for s2 | `0x001` (No Change) | `value="Database"` |
| **4** | `__init__` called for s2 | `0x001` | `value="Database"` (Update Skipped) |

### Summary
* **`s1` and `s2` are the exact same variable.** (`s1 is s2` returns `True`).
* **The Initialization Guard** (`if self._initialized: return`) is the most important part. Without it, the second call would overwrite the data of the first call, breaking the Singleton's state consistency.

# The Decorator Approach

We can create a decorator that wraps a class. The decorator maintains a dictionary of instances.

In [7]:
from functools import wraps

def singleton(cls):
    instances = {}
    
    @wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    
    return get_instance

@singleton
class Logger:
    def __init__(self):
        print("Logger Initialized")

# Usage
l1 = Logger()
l2 = Logger()
# "Logger Initialized" prints only once.

print(l1 is l2) # True

print(Logger) # it becomes the function

Logger Initialized
True
<function Logger at 0xf9fc4c3f57a0>


Pros:
- Very clean syntax (@singleton).
- __init__ is guaranteed to run only once.

Cons:
- The class is no longer a class; it becomes a function.
- type(Logger) returns <class 'function'>.

# Flow of Execution for Decorator Singleton

This document details the trace of execution when using a `@decorator` to implement the Singleton pattern. 

**Key Concept:** Unlike the Class approach, the "Magic" here happens **immediately when the file is loaded**, not just when objects are created.

---

### Phase 0: Definition Time (The Setup)
**Event:** Python reads the file and encounters the `@singleton` decorator above `class Logger`.

1.  **Decorator Execution:** * Python executes the `singleton` function immediately.
    * It creates a local dictionary variable `instances = {}` inside the function's scope (Closure).
    * It defines the inner wrapper function `get_instance`.

2.  **Class Replacement:** * Python takes the original `class Logger` and passes it as an argument to the decorator.
    * The decorator returns the `get_instance` function.
    * **Crucial Change:** The name `Logger` no longer points to the Class. It now points to the `get_instance` function.

---

### Phase 1: The First Call
**User Code:** `l1 = Logger()`

1.  **Function Call:** * You are actually calling `get_instance()`, not the class constructor directly.
    
2.  **Check Dictionary:** * The wrapper checks the closure variable `instances`.
    * **Query:** Is the original `Logger` class key inside `instances`? -> **NO**.

3.  **Instantiation:**
    * The wrapper calls the *original* class: `cls(*args, **kwargs)`.
    * The original `__init__` runs ("Logger Initialized").
    * The new object (e.g., `Object_A`) is created.

4.  **Storage:** * The wrapper saves `Object_A` into the `instances` dictionary.
    * `instances = { <class Logger>: <Object_A> }`

5.  **Return:** * `Object_A` is returned to `l1`.

---

### Phase 2: The Second Call
**User Code:** `l2 = Logger()`

1.  **Function Call:** * You call `get_instance()` again.

2.  **Check Dictionary:** * The wrapper checks `instances`.
    * **Query:** Is the original `Logger` class key inside `instances`? -> **YES**.

3.  **Skip Instantiation:** * The wrapper **skips** calling `cls()`. 
    * The `__init__` method **does NOT run**. (This is a key difference from the `__new__` approach).

4.  **Return:** * The existing `Object_A` (retrieved from the dictionary) is returned to `l2`.

---

### Visual Trace Summary

| Step | Action | `instances` Dictionary (Closure) | What `Logger` refers to |
| :--- | :--- | :--- | :--- |
| **0** | File Load (Definition) | `{}` (Created) | `get_instance` (Function) |
| **1** | User calls `Logger()` | `{ <Class>: <Object_A> }` | `get_instance` (Function) |
| **2** | `__init__` runs | (Inside Object_A) | `get_instance` (Function) |
| **3** | User calls `Logger()` again | (No Change) | `get_instance` (Function) |
| **4** | `__init__` runs? | **NO** (Skipped entirely) | `get_instance` (Function) |

### Summary
* **Identity:** `l1` and `l2` are identical.
* **Initialization:** Unlike the `__new__` method, the Decorator approach prevents `__init__` from running a second time.
* **Trade-off:** The usage is cleaner, but `Logger` is technically a function now, not a class, which can confuse type-checkers or tools that expect `isinstance(Logger, type)` to be true.

# The Metaclass Approach

A Metaclass controls how a class is created. By overriding the __call__ method of the metaclass, we can intercept the creation of the instance entirely. This is generally considered the most robust implementation in Python.

In [8]:
class SingletonMeta(type):
    """
    The Singleton class can be implemented in different ways in Python. Some
    possible methods include: base class, decorator, metaclass. We will use the
    metaclass because it is best suited for this purpose.
    """
    _instances = {}
    
    # __call__ is executed when you write ClassName()
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            # Create the instance and store it
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    def connect(self):
        return "Connected to DB"

# Usage
db1 = DatabaseConnection()
db2 = DatabaseConnection()

print(db1 is db2) # True

True


# The Metaclass Approach (The Thread-safe Way)

To fix the problem, you have to synchronize threads during the first creation of the Singleton object.

In [12]:
from threading import Lock, Thread


class SingletonMeta(type):
    """
    This is a thread-safe implementation of Singleton.
    """

    _instances = {}

    _lock: Lock = Lock()
    """
    We now have a lock object that will be used to synchronize threads during
    first access to the Singleton.
    """

    def __call__(cls, *args, **kwargs):
        """
        Possible changes to the value of the `__init__` argument do not affect
        the returned instance.
        """
        # Now, imagine that the program has just been launched. Since there's no
        # Singleton instance yet, multiple threads can simultaneously pass the
        # previous conditional and reach this point almost at the same time. The
        # first of them will acquire lock and will proceed further, while the
        # rest will wait here.
        with cls._lock:
            # The first thread to acquire the lock, reaches this conditional,
            # goes inside and creates the Singleton instance. Once it leaves the
            # lock block, a thread that might have been waiting for the lock
            # release may then enter this section. But since the Singleton field
            # is already initialized, the thread won't create a new object.
            if cls not in cls._instances:
                instance = super().__call__(*args, **kwargs)
                cls._instances[cls] = instance
        return cls._instances[cls]


class DatabaseConnection(metaclass=SingletonMeta):

    def __init__(self, db: str):
        self.db = db 
        
    def connect(self):
        return f"Connected to {self.db}"


def test_singleton(value: str) -> None:
    conn = DatabaseConnection(value)
    print(conn.connect())


if __name__ == "__main__":
    # The client code.

    print("If you see the same value, then singleton was reused (yay!)\n"
          "If you see different values, "
          "then 2 singletons were created (booo!!)\n\n"
          "RESULT:\n")

    process1 = Thread(target=test_singleton, args=("mysql",))
    process2 = Thread(target=test_singleton, args=("postgres",))
    process1.start()
    process2.start()

If you see the same value, then singleton was reused (yay!)
If you see different values, then 2 singletons were created (booo!!)

RESULT:

Connected to mysql
Connected to mysql


# Flow of Execution for Metaclass Singleton

This document details the trace of execution when using a `metaclass` to control object creation.

**Key Concept:** In Python, a **Class** is an instance of a **Metaclass**.
Therefore, when you "call" a class (e.g., `Database()`), you are actually triggering the `__call__` method of its Metaclass.

---

### Phase 0: Definition Time (The Setup)
**Event:** Python reads the file and parses the class definition `class Database(metaclass=SingletonMeta)`.

1.  **Class Creation:** * Python does not create a standard class. It sees the `metaclass` argument.
2.  **Metaclass Instantiation:** * Python asks `SingletonMeta` to create the `Database` class.
3.  **Storage:** * The `SingletonMeta` class now holds a dictionary `_instances = {}`.
    * The symbol `Database` is now an instance of `SingletonMeta`.

---

### Phase 1: The First Call
**User Code:** `db1 = Database()`

1.  **Metaclass Interception:** * Because `Database` is an instance of `SingletonMeta`, the call triggers `SingletonMeta.__call__`.
    
2.  **Check Dictionary:** * The metaclass checks its internal `_instances` dictionary.
    * **Query:** Is the `Database` class key present? -> **NO**.

3.  **Delegation (Creation):**
    * The metaclass calls `super().__call__()` (which delegates to the standard Python `type` behavior).
    * This standard behavior triggers `Database.__new__` (memory allocation).
    * This standard behavior triggers `Database.__init__` (initialization).
    * A new object (e.g., `Object_A`) is created and fully initialized.

4.  **Storage:** * The metaclass takes `Object_A` and stores it in `_instances`.
    * `_instances = { <class Database>: <Object_A> }`

5.  **Return:** * `Object_A` is returned to `db1`.

---

### Phase 2: The Second Call
**User Code:** `db2 = Database()`

1.  **Metaclass Interception:** * Again, `SingletonMeta.__call__` is triggered.

2.  **Check Dictionary:** * The metaclass checks `_instances`.
    * **Query:** Is the `Database` class key present? -> **YES**.

3.  **Short Circuit:** * The metaclass sees the object exists.
    * **Crucial Action:** It **DOES NOT** call `super().__call__()`.

4.  **Skip Initialization:** * Because `super().__call__` was never invoked, the `Database.__new__` and `Database.__init__` methods are **NEVER TOUCHED**.

5.  **Return:** * The existing `Object_A` is returned to `db2`.

---

### Visual Trace Summary

| Step | Action | Controlled By | Result |
| :--- | :--- | :--- | :--- |
| **0** | Definition | Python Internals | `Database` is created as an instance of `SingletonMeta`. |
| **1** | User calls `Database()` | `SingletonMeta.__call__` | Checks cache. Empty. |
| **2** | Creation | `super().__call__` | Runs `__new__` and `__init__`. |
| **3** | Storage | `SingletonMeta` | Saves instance to dict. |
| **4** | User calls `Database()` | `SingletonMeta.__call__` | Checks cache. **Found.** |
| **5** | Return | `SingletonMeta` | Returns cached object. `__init__` **SKIPPED**. |

### Summary
* **The "Magic":** You intercept the creation process *before* the class itself even gets involved.
* **Efficiency:** Like the Decorator approach, this prevents `__init__` from running twice.
* **Correctness:** Unlike the Decorator approach, `Database` remains a true Class type (not a function wrapper), preserving inheritance and `isinstance` checks.

# The "Pythonic" Way (Modules)

This is the most common way to do Singletons in Python.

Python modules are singletons by design. When you import a module, Python executes it once and caches it in `sys.modules`. Subsequent imports return the same object.

File: `config.py`
```py
# Just define your variables and functions here
api_key = "12345"
debug_mode = True

def do_something():
    print("Doing something")
```

File: main.py
```py
import config
import config as c2

print(config is c2) # True
config.debug_mode = False
print(c2.debug_mode) # False
```

Pros:
- Simplest: Zero boilerplate code.
- Native: Uses Python's internal import mechanics.
- Thread Safe: Python imports are thread-safe.

Cons:
- Not a class (no inheritance, no properties).
- Can't be instantiated lazily (runs on import).
- Harder to test (requires reloading modules to reset state).


#### Summary: Which one should I use?

- Use The Module approach (Method 4) for 95% of cases (Settings, Configuration, Shared State).
- Use The Metaclass approach (Method 3) if you absolutely need a Class (e.g., you need inheritance, properties, or to pass the object around) and you want to enforce the Singleton rule strictly.
- Avoid Method 1 `(__new__)` unless you enjoy debugging why your `__init__` is running multiple times.

# Singleton Design Pattern 

explained using the "Best" Pythonic approach (using a Metaclass) applied to a complex real-world scenario: A Thread-Safe Database Connection Pool.

#### The Scenario: Database Connection Pool
In a high-traffic web application (like a Flask or Django app), you cannot open a new physical connection to the database for every single user request. It is too slow and will crash the database server (e.g., "Too many connections" error).

Instead, you create a **Connection Pool** once when the application starts. This pool maintains, say, 10 open connections and reuses them. 

**The Constraint**: You must ensure there is exactly one instance of the Connection Pool Manager in your entire application. If you accidentally create two pools, you double the load on your database and risk data corruption.

## The Pythonic Implementation (Metaclass Strategy)

In Java, you would write a `public static getInstance()` method and check `if (instance == null)`. In Python, the most robust and "Pro" way to implement a Singleton is using a **Metaclass**. This separates the Singleton logic (the pattern) from the Business logic (the database code).

This implementation handles Thread Safety (essential for web apps) and uses Python's `super()` and `__call__` magic methods.

#### THE SINGLETON METACLASS (The Reusable Logic)

In [1]:
import threading
from typing import Type, Dict

class ThreadSafeSingleton(type):
    """
    A Metaclass that creates a Singleton.
    Any class that uses this as a metaclass will behave as a Singleton.
    """
    _instances: Dict[Type, object] = {}
    _lock: threading.Lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        # Double-Checked Locking Pattern for Thread Safety
        # 1. Fast check: If instance exists, return it immediately (no lock needed)
        if cls not in cls._instances:
            with cls._lock:
                # 2. Slow check: Ensure no other thread created it while we waited for lock
                if cls not in cls._instances:
                    # Create the instance using the parent's logic
                    instance = super().__call__(*args, **kwargs)
                    cls._instances[cls] = instance
        return cls._instances[cls]

#### THE REAL WORLD CLASS (Database Pool)

In [6]:
import time
import random

class DatabasePool(metaclass=ThreadSafeSingleton):
    """
    This class manages a simulated pool of database connections.
    Initialization is expensive (simulated delay).
    """
    def __init__(self, connection_string: str, max_connections: int = 5):
        # This print proves __init__ runs only ONCE
        print(f"🔌 [System] Initializing Database Pool for: {connection_string}")
        
        # Simulate heavy initialization (e.g., verifying credentials, allocating memory)
        time.sleep(1) 
        
        self.connection_string = connection_string
        self.max_connections = max_connections
        self.active_connections = 0

    def get_connection(self):
        if self.active_connections < self.max_connections:
            self.active_connections += 1
            return f"Conn-{random.randint(1000, 9999)}"
        else:
            raise Exception("❌ Connection Pool Exhausted!")

#### CLIENT CODE (Simulating Multi-Threaded App)

import threading

def worker_task(thread_id):
    # Every thread tries to instantiate the class "DatabasePool"
    # But because of the Singleton Metaclass, they all get the SAME object.
    db = DatabasePool("postgresql://prod-db:5432")
    
    try:
        conn = db.get_connection()
        print(f"   👤 Thread-{thread_id} got {conn} (Active: {db.active_connections})")
    except Exception as e:
        print(f"   ⚠️ Thread-{thread_id} Failed: {e}")

def main():
    print("--- Starting Multithreaded Database Application ---")
    
    # We spawn 10 threads roughly at the same time
    threads = []
    for i in range(10):
        t = threading.Thread(target=worker_task, args=(i,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    # Verify that we are indeed using the same instance
    db1 = DatabasePool("postgresql://prod-db:5432")
    db2 = DatabasePool("postgresql://prod-db:5432")
    
    print("\n--- Verification ---")
    print(f"Are db1 and db2 the same object? {'✅ YES' if db1 is db2 else '❌ NO'}")

if __name__ == "__main__":
    main()

#### Why this is the "Best" Pythonic Example?

- Metaclass (`type`) vs Decorator:
    - Using a Decorator (`@singleton`) to wrap a class creates a function wrapper, which messes up inheritance and type-checking (e.g., `isinstance(my_obj, DatabasePool)` might fail).
    - Using a Metaclass `(class DatabasePool(metaclass=...)`) preserves the class nature of the object. db1 is truly an instance of `DatabasePool`.

- Thread Safety (`_lock`):
    - In a real "Software Industry" environment (Gunicorn, uWSGI), race conditions during startup are real. If two threads hit `DatabasePool()` at the exact same millisecond, a naive implementation would create two connections. The `with cls._lock:` block prevents this.

- Separation of Concerns:
    - The DatabasePool class doesn't contain a messy `if instance is None` check in its `__init__`. It focuses purely on database logic.
    - The `ThreadSafeSingleton` metaclass handles the "Pattern" logic. You can reuse this metaclass for `Logger`, `ConfigLoader`, or `CacheManager` easily.